# Neo4j Legal Search

Notebook nay chi dung de nhan prompt, query Neo4j va in ket qua. Khong doc file JSON local.

In [4]:
%pip install langchain_core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import json
import os
import re
import unicodedata
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

candidate_env_paths = [Path('.env'), Path('chatbot_rag/.env')]
for env_path in candidate_env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break
else:
    load_dotenv(override=True)

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

if not NEO4J_URI or not NEO4J_USER or not NEO4J_PASSWORD:
    raise ValueError('Thieu bien moi truong NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD')
if not OPENAI_API_KEY:
    raise ValueError('Thieu bien moi truong OPENAI_API_KEY')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0, api_key=OPENAI_API_KEY)

print(f'Connected to Neo4j at {NEO4J_URI}')
print(f'OpenAI model: {OPENAI_MODEL}')


Connected to Neo4j at neo4j+s://f265c586.databases.neo4j.io
OpenAI model: gpt-4o-mini


In [6]:
GENERIC_TERMS = {
    'toi pham', 'toi danh', 'hinh phat', 'luat hinh su', 'bo luat hinh su',
    'trach nhiem hinh su', 'truy cuu trach nhiem', 'tham nhung'
}
GENERAL_ARTICLES = {17, 55, 58}
GENERAL_CRIME_HINTS = {'dong pham', 'quyet dinh hinh phat', 'tong hop hinh phat'}

class SearchHints(BaseModel):
    crime_hint: str = Field(default='')
    condition_hint: str = Field(default='')
    search_terms: list[str] = Field(default_factory=list)

def normalize_text(text):
    text = (text or '').lower().strip()
    text = text.replace('đ', 'd').replace('Đ', 'D')
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def score_text(term, text, exact_bonus, partial_bonus):
    if not term or not text:
        return 0
    if term == text:
        return exact_bonus
    if term in text or text in term:
        return partial_bonus
    return 0

def safe_number(value, default=0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def parse_money_value(prompt):
    text = normalize_text(prompt)
    compact = text.replace(' ', '')
    m = re.search(r'(\d[\d\.]*)ty', compact)
    if m:
        return float(m.group(1).replace('.', '')) * 1_000_000_000
    m = re.search(r'(\d[\d\.]*)trieu', compact)
    if m:
        return float(m.group(1).replace('.', '')) * 1_000_000
    m = re.search(r'(\d[\d\.]*)nghin', compact)
    if m:
        return float(m.group(1).replace('.', '')) * 1_000
    nums = re.findall(r'\d[\d\.]*', prompt)
    if nums:
        return float(nums[-1].replace('.', ''))
    return None

def extract_prompt_terms(prompt):
    text = normalize_text(prompt)
    phrases = []
    for token in [
        'tai san nha nuoc', 'that thoat', 'lang phi', 'vu khi quan dung',
        'dong vat', 'quy hiem', 'giet nguoi', 'ma tuy', 'hoi lo',
        'phan boi to quoc', 'an ninh quoc gia', 'hiep dam'
    ]:
        if token in text:
            phrases.append(token)
    return phrases

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Chi tra ve JSON hop le.'),
    ('user',
        'Ban la tro ly phap ly. Hay tra ve JSON hop le voi 3 truong sau:\n'
        '- crime_hint: ten toi danh gan nhat theo ngon ngu phap ly\n'
        '- condition_hint: tinh tiet quan trong nhat de tra cuu, uu tien viet ngan gon theo cach dien dat cua dieu luat\n'
        '- search_terms: mang 3 den 8 cum tu khoa ngan phuc vu tim kiem\n'
        'Chi tra ve JSON, khong giai thich.\n\n'
        'Cau cua nguoi dung: {question}'
    )
])
extract_chain = prompt | llm.with_structured_output(SearchHints)

FETCH_ROWS_QUERY = '''
MATCH (cr:Crime)-[:HAS_RULE]->(r:Rule)
OPTIONAL MATCH (r)-[:HAS_PENALTY]->(p:Penalty)
OPTIONAL MATCH (r)-[:HAS_CONDITION]->(c:Condition)
WITH cr, r, p, collect(DISTINCT c.text) AS conditions
RETURN cr.id AS crime_id,
       cr.name AS crime_name,
       cr.article AS article,
       r.id AS rule_id,
       r.clause AS clause,
       r.logic AS logic,
       r.priority AS priority,
       conditions,
       p.min AS penalty_min,
       p.max AS penalty_max,
       p.extra AS penalty_extra,
       p.note AS penalty_note
'''

def fetch_rows():
    with driver.session() as session:
        return [dict(record) for record in session.run(FETCH_ROWS_QUERY)]

def pick_primary_rows(rows):
    specific_rows = [row for row in rows if safe_number(row.get('article'), 0) not in GENERAL_ARTICLES]
    return specific_rows or rows

def retrieve(question, top_k=5):
    hints = extract_chain.invoke({'question': question})
    search_terms = list(hints.search_terms or [])
    search_terms.extend([hints.crime_hint, hints.condition_hint])
    search_terms = [normalize_text(item) for item in search_terms if item and normalize_text(item)]
    search_terms = [item for item in dict.fromkeys(search_terms) if item not in GENERIC_TERMS and len(item) >= 4]
    crime_hint_norm = normalize_text(hints.crime_hint)
    condition_hint_norm = normalize_text(hints.condition_hint)
    amount_value = parse_money_value(question)
    prompt_terms = extract_prompt_terms(question)
    rows = fetch_rows()

    scored_rows = []
    for row in rows:
        crime_name_norm = normalize_text(row['crime_name'])
        conditions_norm = [normalize_text(x) for x in (row.get('conditions') or []) if x]
        article_number = int(safe_number(row.get('article'), 0))
        is_general_article = article_number in GENERAL_ARTICLES
        score = 0
        if crime_hint_norm:
            score += score_text(crime_hint_norm, crime_name_norm, 160, 80)
        for term in prompt_terms:
            score += score_text(term, crime_name_norm, 40, 20)
            score += max([score_text(term, cond, 20, 10) for cond in conditions_norm] or [0])
        for term in search_terms:
            score += score_text(term, crime_name_norm, 12, 6)
            score += max([score_text(term, cond, 10, 5) for cond in conditions_norm] or [0])
        score += max([score_text(condition_hint_norm, cond, 60, 30) for cond in conditions_norm] or [0])
        if crime_hint_norm and crime_hint_norm not in GENERAL_CRIME_HINTS and is_general_article:
            score -= 90
        if 'dong pham' in search_terms and article_number == 17:
            score += 15
        if article_number == 17 and any('hiep dam' in term for term in search_terms) and 'dong pham' not in crime_hint_norm:
            score -= 20
        if amount_value is not None and any(str(amount_value).split('.')[0] in cond for cond in conditions_norm):
            score += 40
        if score > 0:
            row['match_score'] = score
            scored_rows.append(row)

    scored_rows.sort(key=lambda item: (
        -safe_number(item.get('match_score'), 0),
        safe_number(item.get('priority'), 999),
        -safe_number(item.get('clause'), 0),
        safe_number(item.get('article'), 999999),
    ))

    top_rows = pick_primary_rows(scored_rows)[:top_k]
    explanation = None
    if top_rows:
        top = top_rows[0]
        explanation = (
            f'Theo trường hợp của bạn, bạn đã vi phạm tại Điều {top["article"]}, '
            f'khoản {top["clause"]}, {top["crime_name"]}. '
            f'Bạn sẽ bị {format_penalty_text(top)}.'
        )
    return {
        'hints': hints.model_dump(),
        'search_terms': search_terms,
        'prompt_terms': prompt_terms,
        'amount_value': amount_value,
        'rows': top_rows,
        'explanation': explanation,
    }

def format_penalty_text(row):
    if row.get('penalty_note'):
        return row['penalty_note']
    if row.get('penalty_min') is not None and row.get('penalty_max') is not None:
        penalty_text = f"phạt tù từ {row['penalty_min']} đến {row['penalty_max']} năm"
    elif row.get('penalty_min') is not None:
        penalty_text = f"mức phạt từ {row['penalty_min']}"
    else:
        penalty_text = 'chịu hình phạt theo quy định'
    if row.get('penalty_extra'):
        penalty_text += f", {row['penalty_extra']}"
    return penalty_text


In [7]:
user_prompt = 'Tôi giết 2 người thì bị tội gì'
top_k = 5

result = retrieve(user_prompt, top_k=top_k)

print('Hints tu OpenAI:')
print(json.dumps(result['hints'], ensure_ascii=False, indent=2))
print('\nSearch terms:', result['search_terms'])
print('Prompt terms:', result['prompt_terms'])
print('Amount parsed:', result['amount_value'])

print('\nKet qua query Neo4j:')
for row in result['rows']:
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print(
        f"Điều {row['article']} - {row['crime_name']} | khoản {row['clause']} | "
        f"khung hình phạt: {row.get('penalty_min')} den {row.get('penalty_max')} nam"
        + (f", {row['penalty_extra']}" if row.get('penalty_extra') else '')
        + (f", note: {row['penalty_note']}" if row.get('penalty_note') else '')
        + f" | match_score={row['match_score']}"
    )

print('\nGiai thich de doc:')
print(result['explanation'])


Hints tu OpenAI:
{
  "crime_hint": "Giết người",
  "condition_hint": "Có dấu hiệu giết người với 2 nạn nhân",
  "search_terms": [
    "giết người",
    "tội giết người",
    "hình phạt giết người",
    "tội phạm",
    "luật hình sự",
    "tội danh",
    "nạn nhân"
  ]
}

Search terms: ['giet nguoi', 'toi giet nguoi', 'hinh phat giet nguoi', 'nan nhan', 'co dau hieu giet nguoi voi 2 nan nhan']
Prompt terms: []
Amount parsed: 2.0

Ket qua query Neo4j:
{
  "crime_id": "123",
  "crime_name": "Tội giết người",
  "article": 123,
  "rule_id": "123_r1",
  "clause": 1,
  "logic": "AGGRAVATION",
  "priority": 1,
  "conditions": [
    "Giết 02 người trở lên",
    "Giết người dưới 16 tuổi",
    "Giết phụ nữ mà biết là có thai",
    "Giết người đang thi hành công vụ hoặc vì lý do công vụ của nạn nhân",
    "Giết ông, bà, cha, mẹ, người nuôi dưỡng, thầy giáo, cô giáo của mình",
    "Giết người mà liền trước đó hoặc ngay sau đó lại thực hiện một tội phạm rất nghiêm trọng hoặc tội phạm đặc biệt nghiêm

In [ ]:
driver.close()
print('Closed Neo4j driver')


Closed Neo4j driver
